In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    SimpleRNN,
    LSTM,
    GRU,
    Bidirectional,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)



I0000 00:00:1786363134.150484  438047 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1786363134.298563  438047 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1786363141.382016  438047 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


# new lag features with baseling rolling features

In [2]:
df = pd.read_csv("../Dataset/df.csv")

df["Datetime"] = pd.to_datetime(df["Datetime"])

df["Hour"] = df["Datetime"].dt.hour

df["Day"] = df["Datetime"].dt.day

df["DayOfWeek"] = df["Datetime"].dt.dayofweek

df["Week"] = df["Datetime"].dt.isocalendar().week.astype(int)

df["Month"] = df["Datetime"].dt.month

df["Year"] = df["Datetime"].dt.year

df["IsWeekend"] = ( df["DayOfWeek"] >= 5 ).astype(int)

# Lag 

df["Lag_1"] = df["PJME_MW"].shift(1)
df["Lag_2"] = df["PJME_MW"].shift(2)
df["Lag_3"] = df["PJME_MW"].shift(3)
df["Lag_6"] = df["PJME_MW"].shift(6)
df["Lag_12"] = df["PJME_MW"].shift(12)
df["Lag_24"] = df["PJME_MW"].shift(24)
df["Lag_48"] = df["PJME_MW"].shift(48)
df["Lag_72"] = df["PJME_MW"].shift(72)
df["Lag_168"] = df["PJME_MW"].shift(168)

# Rolling mean and std

df["RollingMean_24"] = (
    df["PJME_MW"]
    .rolling(24)
    .mean()
)


df["RollingMean_168"] = (
    df["PJME_MW"]
    .rolling(168)
    .mean()
)


df["RollingStd_24"] = (
    df["PJME_MW"]
    .rolling(24)
    .std()
)

In [3]:
df

,Datetime,PJME_MW,Hour,DayOfWeek,Month,Day,Week,Year,IsWeekend,Lag_1,...,Lag_3,Lag_6,Lag_12,Lag_24,Lag_48,Lag_72,Lag_168,RollingMean_24,RollingMean_168,RollingStd_24
0,1998-12-31 01:00:00,26498.0,1,3,12,31,53,1998,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1998-12-31 02:00:00,25147.0,2,3,12,31,53,1998,0,26498.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1998-12-31 03:00:00,24574.0,3,3,12,31,53,1998,0,25147.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1998-12-31 04:00:00,24393.0,4,3,12,31,53,1998,0,24574.0,...,26498.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1998-12-31 05:00:00,24860.0,5,3,12,31,53,1998,0,24393.0,...,25147.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145361,2014-03-02 03:00:00,44284.0,3,6,3,2,9,2014,1,44343.0,...,41213.0,38726.0,40154.0,45787.0,43308.0,45896.0,42112.0,40343.500000,41856.327381,2295.270146
145362,2014-03-02 04:00:00,43751.0,4,6,3,2,9,2014,1,44284.0,...,44147.0,38737.0,40309.0,45209.0,42440.0,45377.0,40797.0,40282.750000,41873.910714,2177.148998
145363,2014-03-02 05:00:00,42402.0,5,6,3,2,9,2014,1,43751.0,...,44343.0,39337.0,39884.0,43663.0,40661.0,44092.0,38819.0,40230.208333,41895.238095,2106.081917
145364,2014-03-02 06:00:00,40164.0,6,6,3,2,9,2014,1,42402.0,...,44284.0,41213.0,39544.0,41581.0,38207.0,42257.0,36287.0,40171.166667,41918.315476,2086.336995


In [5]:
df.isna().sum()

Datetime             0
PJME_MW              0
Hour                 0
DayOfWeek            0
Month                0
Day                  0
Week                 0
Year                 0
IsWeekend            0
Lag_1                1
Lag_2                2
Lag_3                3
Lag_6                6
Lag_12              12
Lag_24              24
Lag_48              48
Lag_72              72
Lag_168            168
RollingMean_24      23
RollingMean_168    167
RollingStd_24       23
dtype: int64

In [4]:
missing_count = df.isnull().sum()

missing_percentage = (
    df.isnull().mean() * 100
)

missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percentage": missing_percentage
})

missing_summary

,Missing Count,Missing Percentage
Datetime,0,0.000000
PJME_MW,0,0.000000
Hour,0,0.000000
DayOfWeek,0,0.000000
Month,0,0.000000
Day,0,0.000000
Week,0,0.000000
Year,0,0.000000
IsWeekend,0,0.000000
Lag_1,1,0.000688


In [6]:
df = df.dropna().reset_index(drop=True)

In [7]:
df

,Datetime,PJME_MW,Hour,DayOfWeek,Month,Day,Week,Year,IsWeekend,Lag_1,...,Lag_3,Lag_6,Lag_12,Lag_24,Lag_48,Lag_72,Lag_168,RollingMean_24,RollingMean_168,RollingStd_24
0,1998-12-24 01:00:00,27213.0,1,3,12,24,52,1998,0,28570.0,...,31932.0,32828.0,31652.0,27669.0,26705.0,28445.0,26498.0,29475.375000,30894.589286,2736.646326
1,1998-12-24 02:00:00,25643.0,2,3,12,24,52,1998,0,27213.0,...,30698.0,32670.0,30963.0,26162.0,25977.0,27266.0,25147.0,29453.750000,30897.541667,2765.861628
2,1998-12-24 03:00:00,24907.0,3,3,12,24,52,1998,0,25643.0,...,28570.0,32526.0,30393.0,25483.0,25555.0,26744.0,24574.0,29429.750000,30899.523810,2804.050165
3,1998-12-24 04:00:00,24721.0,4,3,12,24,52,1998,0,24907.0,...,27213.0,31932.0,30230.0,25045.0,25589.0,26760.0,24393.0,29416.250000,30901.476190,2826.766154
4,1998-12-24 05:00:00,25144.0,5,3,12,24,52,1998,0,24721.0,...,25643.0,30698.0,31038.0,25030.0,26047.0,27166.0,24860.0,29421.000000,30903.166667,2819.160745
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145193,2014-03-02 03:00:00,44284.0,3,6,3,2,9,2014,1,44343.0,...,41213.0,38726.0,40154.0,45787.0,43308.0,45896.0,42112.0,40343.500000,41856.327381,2295.270146
145194,2014-03-02 04:00:00,43751.0,4,6,3,2,9,2014,1,44284.0,...,44147.0,38737.0,40309.0,45209.0,42440.0,45377.0,40797.0,40282.750000,41873.910714,2177.148998
145195,2014-03-02 05:00:00,42402.0,5,6,3,2,9,2014,1,43751.0,...,44343.0,39337.0,39884.0,43663.0,40661.0,44092.0,38819.0,40230.208333,41895.238095,2106.081917
145196,2014-03-02 06:00:00,40164.0,6,6,3,2,9,2014,1,42402.0,...,44284.0,41213.0,39544.0,41581.0,38207.0,42257.0,36287.0,40171.166667,41918.315476,2086.336995


In [8]:
test_size = int(len(df) * 0.20)

test_df = df.iloc[-test_size:].copy()

remaining_df = df.iloc[:-test_size].copy()

validation_hours = 60 * 24

val_df = remaining_df.iloc[-validation_hours:].copy()

train_df = remaining_df.iloc[:-validation_hours].copy()

print("Train:")
print(train_df.index.min(), "→", train_df.index.max())
print(train_df.shape)

print("\nValidation:")
print(val_df.index.min(), "→", val_df.index.max())
print(val_df.shape)

print("\nTest:")
print(test_df.index.min(), "→", test_df.index.max())
print(test_df.shape)

Train:
0 → 114718
(114719, 21)

Validation:
114719 → 116158
(1440, 21)

Test:
116159 → 145197
(29039, 21)


In [9]:
scaler = StandardScaler()

train_scaled = scaler.fit_transform( train_df[["PJME_MW"]] )


val_scaled = scaler.transform( val_df[["PJME_MW"]] )


test_scaled = scaler.transform( test_df[["PJME_MW"]] )

In [10]:
def create_sequences(data, sequence_length, forecast_horizon):

    X = []
    y = []

    for i in range(sequence_length, len(data) - forecast_horizon + 1):

        X.append(data[i-sequence_length:i])

        y.append(data[i:i+forecast_horizon])

    return np.array(X), np.array(y)



In [11]:
SEQUENCE_LENGTH = 168
FORECAST_HORIZON = 24


X_train, y_train = create_sequences(
    train_scaled,
    SEQUENCE_LENGTH,
    FORECAST_HORIZON
)

val_input = np.concatenate([
    train_scaled[-SEQUENCE_LENGTH:],
    val_scaled
])


X_val, y_val = create_sequences(
    val_input,
    SEQUENCE_LENGTH,
    FORECAST_HORIZON
)

test_input = np.concatenate([
    val_scaled[-SEQUENCE_LENGTH:],
    test_scaled
])


X_test, y_test = create_sequences(
    test_input,
    SEQUENCE_LENGTH,
    FORECAST_HORIZON
)

In [13]:
def build_bilstm_model(sequence_length):

    model = Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, 1)
        ),

        Bidirectional(
            LSTM(64)
            
        ),

        Dense(64, activation="relu"),

        Dense(24)
    ])

    return model


In [14]:
bilstm_model = build_bilstm_model(
    SEQUENCE_LENGTH
)

bilstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)


history_bilstm = bilstm_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

E0000 00:00:1786363961.415118  438047 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/10


W0000 00:00:1786363962.349430  438047 cpu_allocator_impl.cc:82] Allocation of 76962816 exceeds 10% of free system memory.


  57/1790 ━━━━━━━━━━━━━━━━━━━━ 5:00 173ms/step - loss: 0.5877 - mae: 0.5865

KeyboardInterrupt: 